# Crisis Detector Visualization

Interactive visualization of the crisis detection system.

## Setup

First, compile the shared library:
```bash
gcc -O3 -march=native -fPIC -shared \
    crisis_detector.c event_detector.c sr_detector.c hawkes_integrator.c \
    -lm -o libcrisis.so
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd

# Import our wrapper
from crisis_detector import CrisisDetector, CrisisState, simulate_market

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

## 1. Basic Demo: Flash Crash Detection

In [ ]:
# Simulate market with flash crash
np.random.seed(42)

n_ticks = 3000
sigma_peace = 0.01
sigma_crisis = 0.05

# Normal market with crisis at tick 1000-1150
returns = simulate_market(
    n_ticks=n_ticks,
    sigma_peace=sigma_peace,
    crisis_start=1000,
    crisis_duration=150,
    sigma_crisis=sigma_crisis,
    seed=42
)

print(f"Simulated {n_ticks} ticks")
print(f"True crisis: ticks 1000-1150")
print(f"Normal σ: {sigma_peace}, Crisis σ: {sigma_crisis}")

In [ ]:
# Run crisis detector
cd = CrisisDetector()
states = cd.process_batch(returns)

# Get crisis regions
crisis_regions = cd.get_crisis_regions()

print(f"\nDetected {len(crisis_regions)} crisis region(s):")
for start, end in crisis_regions:
    print(f"  Ticks {start} - {end} (duration: {end - start})")

In [ ]:
# Visualization
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

ticks = np.arange(len(returns))

# Plot 1: Returns
ax1 = axes[0]
ax1.plot(ticks, returns, 'b-', linewidth=0.5, alpha=0.7)

# True crisis region
ax1.axvspan(1000, 1150, alpha=0.2, color='blue', label='True Crisis')

# Detected crisis regions
for i, (start, end) in enumerate(crisis_regions):
    ax1.axvspan(start, end, alpha=0.3, color='red', 
                label='Detected' if i == 0 else None)

ax1.set_ylabel('Returns')
ax1.set_title('Flash Crash Detection')
ax1.legend(loc='upper right')
ax1.set_ylim(-0.2, 0.2)

# Plot 2: Cumulative returns
ax2 = axes[1]
cum_ret = np.cumsum(returns)
ax2.plot(ticks, cum_ret, 'g-', linewidth=1)
ax2.axvspan(1000, 1150, alpha=0.2, color='blue')
for start, end in crisis_regions:
    ax2.axvspan(start, end, alpha=0.3, color='red')
ax2.set_ylabel('Cumulative Returns')

# Plot 3: Rolling volatility
ax3 = axes[2]
window = 50
rolling_vol = pd.Series(returns).rolling(window).std() * np.sqrt(252)
ax3.plot(ticks, rolling_vol, 'purple', linewidth=1)
ax3.axhline(sigma_peace * np.sqrt(252), color='green', linestyle='--', 
            label=f'Peace σ ({sigma_peace*np.sqrt(252):.1%})')
ax3.axhline(sigma_crisis * np.sqrt(252), color='red', linestyle='--',
            label=f'Crisis σ ({sigma_crisis*np.sqrt(252):.1%})')
ax3.axvspan(1000, 1150, alpha=0.2, color='blue')
for start, end in crisis_regions:
    ax3.axvspan(start, end, alpha=0.3, color='red')
ax3.set_ylabel('Rolling Vol (ann.)')
ax3.legend(loc='upper right')

# Plot 4: State
ax4 = axes[3]
colors = {0: '#2ecc71', 1: '#f1c40f', 2: '#e74c3c', 3: '#e67e22'}
state_colors = [colors[s] for s in states]
ax4.scatter(ticks, states, c=state_colors, s=2, alpha=0.6)
ax4.set_ylabel('State')
ax4.set_xlabel('Tick')
ax4.set_yticks([0, 1, 2, 3])
ax4.set_yticklabels(['IDLE', 'ALERT', 'ACTIVE', 'RECOVERING'])

# Legend
patches = [
    mpatches.Patch(color='#2ecc71', label='IDLE'),
    mpatches.Patch(color='#f1c40f', label='ALERT'),
    mpatches.Patch(color='#e74c3c', label='ACTIVE'),
    mpatches.Patch(color='#e67e22', label='RECOVERING'),
]
ax4.legend(handles=patches, loc='upper right', ncol=4)

plt.tight_layout()
plt.savefig('flash_crash_detection.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Multiple Crisis Scenario

In [ ]:
# Multiple crises of varying severity
np.random.seed(123)
n_ticks = 5000

returns = np.random.randn(n_ticks) * 0.01

# Crisis 1: Moderate (3x vol) at tick 500
returns[500:600] = np.random.randn(100) * 0.03

# Crisis 2: Severe (5x vol) at tick 1500
returns[1500:1650] = np.random.randn(150) * 0.05

# Crisis 3: Extreme (8x vol) at tick 2500
returns[2500:2550] = np.random.randn(50) * 0.08

# Crisis 4: Prolonged moderate (3x vol) at tick 3500
returns[3500:3800] = np.random.randn(300) * 0.03

true_crises = [(500, 600), (1500, 1650), (2500, 2550), (3500, 3800)]

print("True crises:")
for start, end in true_crises:
    print(f"  {start}-{end}: {end-start} ticks")

In [ ]:
# Run detector
cd = CrisisDetector()
states = cd.process_batch(returns)
detected = cd.get_crisis_regions()

print(f"\nDetected {len(detected)} crisis region(s):")
for start, end in detected:
    # Check which true crisis this corresponds to
    matching = [t for t in true_crises if t[0] <= end and t[1] >= start]
    print(f"  {start}-{end}: {end-start} ticks", 
          f"(matches true crisis at {matching[0][0]})" if matching else "(false positive)")

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(16, 6))

ticks = np.arange(len(returns))

# Plot returns
ax.plot(ticks, returns, 'b-', linewidth=0.3, alpha=0.7)

# True crises (blue)
for i, (start, end) in enumerate(true_crises):
    ax.axvspan(start, end, alpha=0.15, color='blue', 
               label='True Crisis' if i == 0 else None)

# Detected crises (red border)
for i, (start, end) in enumerate(detected):
    ax.axvspan(start, end, alpha=0.25, color='red', 
               label='Detected' if i == 0 else None)

ax.set_xlabel('Tick')
ax.set_ylabel('Returns')
ax.set_title('Multiple Crisis Detection')
ax.legend(loc='upper right')
ax.set_ylim(-0.25, 0.25)

# Add state bar at bottom
state_ax = ax.twinx()
state_ax.set_ylim(-0.1, 1)
state_ax.set_yticks([])

# Color bar for states
for t, s in enumerate(states):
    if s > 0:  # Only show non-IDLE
        colors = {1: '#f1c40f', 2: '#e74c3c', 3: '#e67e22'}
        state_ax.axvline(t, ymin=0, ymax=0.05, color=colors[s], alpha=0.5)

plt.tight_layout()
plt.savefig('multiple_crises.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Parameter Sensitivity

In [ ]:
# Test different configurations
configs = {
    'Default': {},
    'Sensitive (conf=2)': {'confirmation_ticks': 2, 'min_hold_active': 30},
    'Conservative (conf=5)': {'confirmation_ticks': 5, 'min_hold_active': 100},
}

# Use the multiple crisis data
results = {}

for name, cfg in configs.items():
    cd = CrisisDetector(config=cfg)
    states = cd.process_batch(returns)
    detected = cd.get_crisis_regions()
    results[name] = {
        'states': states,
        'detected': detected,
        'n_crises': len(detected),
    }
    print(f"\n{name}: {len(detected)} crises detected")
    for start, end in detected:
        print(f"  {start}-{end}")

In [ ]:
# Compare configurations
fig, axes = plt.subplots(len(configs), 1, figsize=(16, 3*len(configs)), sharex=True)

for ax, (name, res) in zip(axes, results.items()):
    ax.plot(ticks, returns, 'b-', linewidth=0.3, alpha=0.5)
    
    # True crises
    for start, end in true_crises:
        ax.axvspan(start, end, alpha=0.1, color='blue')
    
    # Detected
    for start, end in res['detected']:
        ax.axvspan(start, end, alpha=0.3, color='red')
    
    ax.set_ylabel('Returns')
    ax.set_title(f"{name}: {res['n_crises']} crises detected")
    ax.set_ylim(-0.25, 0.25)

axes[-1].set_xlabel('Tick')
plt.tight_layout()
plt.savefig('parameter_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Detection Metrics

In [ ]:
def compute_metrics(true_crises, detected, n_ticks):
    """Compute detection metrics."""
    # Create binary arrays
    true_mask = np.zeros(n_ticks, dtype=bool)
    det_mask = np.zeros(n_ticks, dtype=bool)
    
    for start, end in true_crises:
        true_mask[start:end] = True
    for start, end in detected:
        det_mask[start:end] = True
    
    # Metrics
    tp = np.sum(true_mask & det_mask)
    fp = np.sum(~true_mask & det_mask)
    fn = np.sum(true_mask & ~det_mask)
    tn = np.sum(~true_mask & ~det_mask)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Detection delay for each true crisis
    delays = []
    for t_start, t_end in true_crises:
        # Find first detection tick within or after true crisis
        for d_start, d_end in detected:
            if d_start <= t_end and d_end >= t_start:  # Overlap
                delay = max(0, d_start - t_start)
                delays.append(delay)
                break
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'avg_delay': np.mean(delays) if delays else None,
        'true_positives': len([d for d in detected if any(
            t[0] <= d[1] and t[1] >= d[0] for t in true_crises)]),
        'false_positives': len([d for d in detected if not any(
            t[0] <= d[1] and t[1] >= d[0] for t in true_crises)]),
    }

# Compute metrics for each config
print("Detection Metrics:")
print("=" * 70)
for name, res in results.items():
    metrics = compute_metrics(true_crises, res['detected'], len(returns))
    print(f"\n{name}:")
    print(f"  Precision: {metrics['precision']:.2%}")
    print(f"  Recall:    {metrics['recall']:.2%}")
    print(f"  F1 Score:  {metrics['f1']:.2%}")
    print(f"  Avg Delay: {metrics['avg_delay']:.1f} ticks" if metrics['avg_delay'] else "  Avg Delay: N/A")
    print(f"  True Pos:  {metrics['true_positives']}/{len(true_crises)}")
    print(f"  False Pos: {metrics['false_positives']}")

## 5. Interactive Widget (Optional)

Requires `ipywidgets`:
```bash
pip install ipywidgets
```

In [ ]:
try:
    from ipywidgets import interact, IntSlider, FloatSlider
    
    @interact(
        confirmation_ticks=IntSlider(min=1, max=10, value=3, description='Confirm'),
        min_hold_active=IntSlider(min=10, max=200, value=50, step=10, description='Min Hold'),
        cooldown_ticks=IntSlider(min=0, max=300, value=100, step=25, description='Cooldown'),
    )
    def interactive_plot(confirmation_ticks, min_hold_active, cooldown_ticks):
        cd = CrisisDetector(config={
            'confirmation_ticks': confirmation_ticks,
            'min_hold_active': min_hold_active,
            'cooldown_ticks': cooldown_ticks,
        })
        states = cd.process_batch(returns)
        detected = cd.get_crisis_regions()
        
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(np.arange(len(returns)), returns, 'b-', linewidth=0.3, alpha=0.5)
        
        for start, end in true_crises:
            ax.axvspan(start, end, alpha=0.1, color='blue')
        for start, end in detected:
            ax.axvspan(start, end, alpha=0.3, color='red')
        
        metrics = compute_metrics(true_crises, detected, len(returns))
        ax.set_title(f"Detected: {len(detected)} | F1: {metrics['f1']:.2%} | Delay: {metrics['avg_delay']:.0f} ticks" 
                     if metrics['avg_delay'] else f"Detected: {len(detected)} | F1: {metrics['f1']:.2%}")
        ax.set_ylim(-0.25, 0.25)
        plt.show()
        
except ImportError:
    print("ipywidgets not installed. Run: pip install ipywidgets")

## 6. Save Results

In [ ]:
# Export to CSV
cd = CrisisDetector()
states = cd.process_batch(returns)

df = pd.DataFrame({
    'tick': np.arange(len(returns)),
    'return': returns,
    'state': states,
    'state_name': [CrisisState(s).name for s in states],
    'is_crisis': states >= 2,
})

df.to_csv('crisis_detection_results.csv', index=False)
print("Saved to crisis_detection_results.csv")
df.head(10)